 Análise Exploratória — Churn de Clientes Bancários

Objetivo: Entender o perfil dos clientes que cancelam (churn) versus os que permanecem, identificando os principais fatores de risco e oportunidades de retenção.



## 1. Importações e configurações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

# Estilo visual 
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

COLORS = {'permaneceu': '#4C72B0', 'churn': '#DD4949'}
PALETTE = [COLORS['permaneceu'], COLORS['churn']]



## 2. Carregamento e inspeção inicial

In [ ]:
# Ajustei o caminho conforme sua estrutura de pastas
df = pd.read_csv('../data/raw/Churn_Modelling.csv')

print(f'Shape: {df.shape[0]:,} linhas × {df.shape[1]} colunas')
df.head()

In [ ]:
# Visão geral das colunas, tipos e valores não-nulos
df.info()

In [ ]:
# Estatísticas descritivas — primeiros sinais de distribuição e outliers
df.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

In [ ]:
# Verificação de valores nulos
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)

resumo_nulos = pd.DataFrame({
    'Nulos': nulos,
    '% do total': nulos_pct
}).query('Nulos > 0')

if resumo_nulos.empty:
    print('✅ Nenhum valor nulo encontrado!')
else:
    display(resumo_nulos)

In [ ]:
# Verificação de duplicatas
n_duplicatas = df.duplicated().sum()
print(f'Linhas duplicadas: {n_duplicatas}')

# Colunas que não agregam valor analítico
print(f'\nColunas a remover: RowNumber, CustomerId, Surname')
df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])
print(f'Shape após limpeza: {df.shape}')

## 3. Limpeza e padronização

In [ ]:
# Renomeei colunas para português (melhora legibilidade do notebook)
df = df.rename(columns={
    'CreditScore'    : 'score_credito',
    'Geography'      : 'pais',
    'Gender'         : 'genero',
    'Age'            : 'idade',
    'Tenure'         : 'anos_cliente',
    'Balance'        : 'saldo',
    'NumOfProducts'  : 'num_produtos',
    'HasCrCard'      : 'tem_cartao',
    'IsActiveMember' : 'membro_ativo',
    'EstimatedSalary': 'salario_estimado',
    'Exited'         : 'churn'
})

# Criei coluna legível para visualizações
df['churn_label'] = df['churn'].map({0: 'Permaneceu', 1: 'Churn'})

# Faixas etárias 
bins = [17, 30, 40, 50, 60, 100]
labels = ['18-30', '31-40', '41-50', '51-60', '60+']
df['faixa_etaria'] = pd.cut(df['idade'], bins=bins, labels=labels)

# Faixas de saldo
df['faixa_saldo'] = pd.cut(
    df['saldo'],
    bins=[-1, 0, 50000, 100000, 150000, 300000],
    labels=['Zero', 'Baixo', 'Médio', 'Alto', 'Muito Alto']
)


df.head(3)

## 4. Análise univariada



In [ ]:
# ── Distribuição da variável alvo ──────────────────────────────────────────
contagem = df['churn'].value_counts()
pct_churn = contagem[1] / len(df) * 100

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Barras
axes[0].bar(['Permaneceu', 'Churn'], contagem.values, color=PALETTE, width=0.5)
for i, v in enumerate(contagem.values):
    axes[0].text(i, v + 80, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=11)
axes[0].set_title('Distribuição de Churn', fontweight='bold')
axes[0].set_ylabel('Número de clientes')

# Pizza
axes[1].pie(contagem.values, labels=['Permaneceu', 'Churn'],
            colors=PALETTE, autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proporção Churn vs Permaneceu', fontweight='bold')

plt.suptitle(f'Taxa de churn: {pct_churn:.1f}% dos clientes', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../reports/01_distribuicao_churn.png', bbox_inches='tight')
plt.show()

print(f'⚠️  Dataset desbalanceado: apenas {pct_churn:.1f}% de churn.')


In [ ]:
# ── Distribuição das variáveis numéricas ───────────────────────────────────
numericas = ['score_credito', 'idade', 'anos_cliente', 'saldo',
             'num_produtos', 'salario_estimado']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(numericas):
    skew = df[col].skew()
    sns.histplot(df[col], ax=axes[i], kde=True, color='#4C72B0', bins=30)
    axes[i].set_title(f'{col}\n(assimetria: {skew:.2f})', fontsize=10)
    axes[i].set_xlabel('')

plt.suptitle('Distribuição das Variáveis Numéricas', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/02_distribuicao_numericas.png', bbox_inches='tight')
plt.show()

print('📝 Observação: saldo tem distribuição bimodal — muitos clientes com saldo zero.')
print('   Isso é um sinal importante para análise de churn.')

In [ ]:
# ── Distribuição das variáveis categóricas ─────────────────────────────────
categoricas = ['pais', 'genero', 'num_produtos', 'tem_cartao', 'membro_ativo']

fig, axes = plt.subplots(1, len(categoricas), figsize=(18, 4))

for i, col in enumerate(categoricas):
    contagem_col = df[col].value_counts()
    axes[i].bar(contagem_col.index.astype(str), contagem_col.values,
                color='#4C72B0', width=0.5)
    axes[i].set_title(col, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=15)
    for j, v in enumerate(contagem_col.values):
        axes[i].text(j, v + 30, f'{v:,}', ha='center', fontsize=9)

plt.suptitle('Distribuição das Variáveis Categóricas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/03_distribuicao_categoricas.png', bbox_inches='tight')
plt.show()

## 5. Análise bivariada — Churn vs cada variável



In [ ]:
# ── Função auxiliar: taxa de churn por categoria ───────────────────────────
def plot_churn_rate(coluna, titulo, ax, rotacao=0, ordenar=True):
    """Plota taxa de churn (%) por categoria com barras e anotações."""
    taxa = df.groupby(coluna)['churn'].mean().mul(100).round(1)
    if ordenar:
        taxa = taxa.sort_values(ascending=False)
    
    bars = ax.bar(taxa.index.astype(str), taxa.values,
                  color=[COLORS['churn'] if v > df['churn'].mean()*100
                         else COLORS['permaneceu'] for v in taxa.values],
                  width=0.5)
    
    # Linha de referência = média geral
    media_geral = df['churn'].mean() * 100
    ax.axhline(media_geral, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    ax.text(len(taxa) - 0.5, media_geral + 0.5, f'Média: {media_geral:.1f}%',
            fontsize=9, color='gray')
    
    for bar, val in zip(bars, taxa.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')
    
    ax.set_title(titulo, fontweight='bold')
    ax.set_ylabel('Taxa de churn (%)')
    ax.tick_params(axis='x', rotation=rotacao)
    ax.set_ylim(0, taxa.max() * 1.2)

print('✅ Função auxiliar definida!')

In [ ]:
# ── Taxa de churn por variáveis categóricas ────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

plot_churn_rate('pais',         'Churn por País',              axes[0,0])
plot_churn_rate('genero',       'Churn por Gênero',            axes[0,1])
plot_churn_rate('num_produtos', 'Churn por Nº de Produtos',    axes[0,2], ordenar=False)
plot_churn_rate('faixa_etaria', 'Churn por Faixa Etária',      axes[1,0], ordenar=False)
plot_churn_rate('membro_ativo', 'Churn por Membro Ativo',      axes[1,1], ordenar=False)
plot_churn_rate('faixa_saldo',  'Churn por Faixa de Saldo',    axes[1,2], ordenar=False)

plt.suptitle('Taxa de Churn por Perfil de Cliente', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/04_churn_por_categoria.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Distribuição de variáveis numéricas: Churn vs Permaneceu ───────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(numericas):
    for label, color in zip([0, 1], PALETTE):
        subset = df[df['churn'] == label][col]
        sns.kdeplot(subset, ax=axes[i], color=color,
                    fill=True, alpha=0.35, linewidth=1.5,
                    label='Permaneceu' if label == 0 else 'Churn')
    
    # Teste estatístico: as médias são diferentes?
    g0 = df[df['churn']==0][col]
    g1 = df[df['churn']==1][col]
    stat, pval = stats.mannwhitneyu(g0, g1, alternative='two-sided')
    sig = '✅ sig.' if pval < 0.05 else '❌ não sig.'
    
    axes[i].set_title(f'{col}\np-valor: {pval:.4f} {sig}', fontsize=10)
    axes[i].legend(fontsize=9)
    axes[i].set_xlabel('')

plt.suptitle('Distribuição das Variáveis Numéricas: Churn vs Permaneceu',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/05_distribuicao_por_churn.png', bbox_inches='tight')
plt.show()

print('📝 Mann-Whitney U: teste não-paramétrico robusto para comparar distribuições.')
print('   p < 0.05 indica que a variável tem distribuição diferente entre os grupos.')

In [ ]:
# ── Boxplots: visualizar mediana e outliers por grupo ─────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(numericas):
    sns.boxplot(data=df, x='churn_label', y=col, ax=axes[i],
                palette=PALETTE, width=0.4,
                order=['Permaneceu', 'Churn'],
                flierprops=dict(marker='o', markersize=2, alpha=0.3))
    
    # Anotar medianas
    for j, label in enumerate(['Permaneceu', 'Churn']):
        mediana = df[df['churn_label']==label][col].median()
        axes[i].text(j, mediana, f' {mediana:,.0f}',
                     va='center', fontsize=9, fontweight='bold')
    
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('')

plt.suptitle('Boxplots: Comparação de Medianas por Grupo',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/06_boxplots_por_churn.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Análise cruzada: Idade × País × Churn ────────────────────────────────
# Cruzar duas variáveis com churn revela padrões mais ricos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap: churn por faixa etária e país
pivot = df.groupby(['faixa_etaria', 'pais'])['churn'].mean().mul(100).round(1).unstack()
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn_r',
            ax=axes[0], linewidths=0.5, vmin=0, vmax=50,
            cbar_kws={'label': 'Taxa de churn (%)'})
axes[0].set_title('Taxa de Churn (%) — Faixa Etária × País', fontweight='bold')

# Heatmap: churn por número de produtos e membro ativo
pivot2 = df.groupby(['num_produtos', 'membro_ativo'])['churn'].mean().mul(100).round(1).unstack()
pivot2.columns = ['Inativo', 'Ativo']
sns.heatmap(pivot2, annot=True, fmt='.1f', cmap='RdYlGn_r',
            ax=axes[1], linewidths=0.5, vmin=0, vmax=100,
            cbar_kws={'label': 'Taxa de churn (%)'})
axes[1].set_title('Taxa de Churn (%) — Nº Produtos × Atividade', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/07_heatmaps_cruzados.png', bbox_inches='tight')
plt.show()

print('📝 Heatmaps revelam combinações de risco que gráficos simples escondem.')

## 6. Análise de correlações

In [ ]:
# ── Correlação de Pearson entre variáveis numéricas ────────────────────────
cols_corr = ['score_credito', 'idade', 'anos_cliente', 'saldo',
             'num_produtos', 'tem_cartao', 'membro_ativo', 'salario_estimado', 'churn']

corr_matrix = df[cols_corr].corr()

fig, ax = plt.subplots(figsize=(10, 8))

# Máscara para triângulo superior (evita redundância)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            ax=ax, linewidths=0.5, square=True,
            cbar_kws={'shrink': 0.8})

ax.set_title('Matriz de Correlação de Pearson', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/08_correlacao_pearson.png', bbox_inches='tight')
plt.show()

# Destacar correlações mais fortes com churn
print('\n📊 Correlação com churn (ordenado):')
print(corr_matrix['churn'].drop('churn').sort_values(key=abs, ascending=False).to_string())

In [ ]:
# ── Point-Biserial: correlação de variáveis contínuas com churn (binário) ─
# Mais adequado que Pearson quando a variável alvo é binária
from scipy.stats import pointbiserialr

cols_continuas = ['score_credito', 'idade', 'anos_cliente', 'saldo', 'salario_estimado']

resultados = []
for col in cols_continuas:
    r, p = pointbiserialr(df['churn'], df[col])
    resultados.append({'variavel': col, 'correlacao': round(r, 4), 'p_valor': round(p, 6)})

pb_df = pd.DataFrame(resultados).sort_values('correlacao', key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
colors = [COLORS['churn'] if v > 0 else COLORS['permaneceu'] for v in pb_df['correlacao']]
bars = ax.barh(pb_df['variavel'], pb_df['correlacao'], color=colors, height=0.4)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlação Point-Biserial com Churn', fontweight='bold')
ax.set_xlabel('Coeficiente de correlação')
for bar, val in zip(bars, pb_df['correlacao']):
    ax.text(val + (0.002 if val >= 0 else -0.002), bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('../reports/09_correlacao_pointbiserial.png', bbox_inches='tight')
plt.show()

print(pb_df.to_string(index=False))

## 7. Insights e conclusões de negócio


In [ ]:
# ── Resumo quantitativo dos principais achados ─────────────────────────────
taxa_geral   = df['churn'].mean() * 100
taxa_alemanha = df[df['pais']=='Germany']['churn'].mean() * 100
taxa_franca  = df[df['pais']=='France']['churn'].mean() * 100
taxa_feminino = df[df['genero']=='Female']['churn'].mean() * 100
taxa_masculino = df[df['genero']=='Male']['churn'].mean() * 100
taxa_4prod   = df[df['num_produtos']==4]['churn'].mean() * 100
taxa_inativo = df[df['membro_ativo']==0]['churn'].mean() * 100
taxa_ativo   = df[df['membro_ativo']==1]['churn'].mean() * 100
idade_churn  = df[df['churn']==1]['idade'].median()
idade_perm   = df[df['churn']==0]['idade'].median()

print('='*60)
print('         RESUMO EXECUTIVO — EDA CHURN BANCÁRIO')
print('='*60)
print(f'\n📌 Taxa geral de churn: {taxa_geral:.1f}%')
print(f'\n🌍 Churn por país:')
print(f'   Alemanha: {taxa_alemanha:.1f}% ({taxa_alemanha/taxa_geral:.1f}x acima da média)')
print(f'   França:   {taxa_franca:.1f}%')
print(f'\n👤 Churn por gênero:')
print(f'   Feminino:  {taxa_feminino:.1f}% — risco {taxa_feminino/taxa_masculino:.1f}x maior que masculino')
print(f'   Masculino: {taxa_masculino:.1f}%')
print(f'\n🎂 Idade mediana:')
print(f'   Clientes em churn:     {idade_churn:.0f} anos')
print(f'   Clientes retidos:      {idade_perm:.0f} anos')
print(f'   → Clientes mais velhos têm maior propensão ao churn')
print(f'\n📦 Clientes com 4 produtos: taxa de {taxa_4prod:.1f}% de churn (crítico!)')
print(f'\n🔴 Membros inativos: {taxa_inativo:.1f}% churn')
print(f'   Membros ativos:   {taxa_ativo:.1f}% churn')
print(f'   → Atividade é o melhor preditor comportamental encontrado')
print('='*60)

In [ ]:
# ── Gráfico final: painel de insights 
fig = plt.figure(figsize=(14, 8))
fig.suptitle('Principais Drivers de Churn — Resumo Executivo',
             fontsize=15, fontweight='bold', y=1.01)

# Dados dos insights
insights = {
    'País'       : df.groupby('pais')['churn'].mean().mul(100),
    'Gênero'     : df.groupby('genero')['churn'].mean().mul(100),
    'Atividade'  : df.groupby('membro_ativo')['churn'].mean().mul(100).rename({0:'Inativo',1:'Ativo'}),
    'Faixa etária': df.groupby('faixa_etaria')['churn'].mean().mul(100),
}

posicoes = [(0,0), (0,1), (1,0), (1,1)]
gs = fig.add_gridspec(2, 2, hspace=0.4, wspace=0.3)

for (r, c), (titulo, dados) in zip(posicoes, insights.items()):
    ax = fig.add_subplot(gs[r, c])
    cores = [COLORS['churn'] if v > taxa_geral else COLORS['permaneceu']
             for v in dados.values]
    bars = ax.bar(dados.index.astype(str), dados.values, color=cores, width=0.5)
    ax.axhline(taxa_geral, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    ax.text(len(dados)-0.5, taxa_geral+0.5, f'Média: {taxa_geral:.1f}%',
            fontsize=8, color='gray', ha='right')
    for bar, val in zip(bars, dados.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')
    ax.set_title(titulo, fontweight='bold')
    ax.set_ylabel('Churn (%)')
    ax.set_ylim(0, dados.max() * 1.3)

plt.savefig('../reports/10_painel_executivo.png', bbox_inches='tight', dpi=150)
plt.show()


In [ ]:

df.to_csv('../data/processed/churn_eda_completo.csv', index=False)
print('✅ Dataset processado salvo em data/processed/churn_eda_completo.csv')
print(f'   Shape final: {df.shape}')
print(f'   Colunas: {list(df.columns)}')